# Notebook 07 — Policy Simulation, Risk Score & Final Knowledge Layer

**Member 3 tasks C, D, E** — Compute the Composite Risk Priority Score, run the policy simulations, and produce the final `knowledge_layer.csv` that Member 4's chatbot reads.

> **This notebook is the critical handover gate.** Nothing that isn't in `knowledge_layer.csv` can appear in the chatbot.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

from src.data_io import PROCESSED_DIR

FIGURES_DIR = Path('..') / 'reports' / 'figures' / 'clustering'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## 1. Load the interim knowledge layer (with cluster labels)

In [ ]:
kl = pd.read_csv(PROCESSED_DIR / 'knowledge_layer_interim.csv')
print(f"Interim shape: {kl.shape}")
print("cluster_label counts:", kl['cluster_label'].value_counts().to_dict())
assert 'cluster_label' in kl.columns, "Run notebook 05 first!"
kl.head(3)

## 2. Composite Risk Priority Score (0–100)

### 2a. Weight specification and rationale

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RISK PRIORITY SCORE — weight rationale
# ─────────────────────────────────────────────────────────────────────────────
# The score is designed to prioritise enforcement attention.
# Four dimensions; weights sum to 1.0:
#
#   str_density          (0.20) — raw listing count per neighbourhood.
#                                  Larger footprint = larger potential impact,
#                                  but alone it doesn't indicate displacement.
#
#   entire_home_share    (0.30) — fraction of listings that are entire homes.
#                                  HIGHEST weight because entire-home listings
#                                  are the most credible proxy for residential
#                                  units removed from the long-term market.
#
#   commercial_host_share (0.20) — fraction from multi-listing / commercial hosts.
#                                  Signals professionalised STR activity that
#                                  is harder to regulate than casual hosting.
#
#   breach_rate_90       (0.30) — fraction of entire-home listings exceeding
#                                  90 booked nights/year (London cap proxy).
#                                  HIGHEST weight alongside entire_home_share
#                                  because it directly measures regulatory
#                                  non-compliance risk.
# ─────────────────────────────────────────────────────────────────────────────

WEIGHTS = {
    'str_density':            0.20,
    'entire_home_share':      0.30,
    'commercial_host_share':  0.20,
    'breach_rate_90':         0.30,
}

print("Weights:")
for k, v in WEIGHTS.items():
    print(f"  {k}: {v}")
print(f"  Sum: {sum(WEIGHTS.values())}")

### 2b. Compute the score

In [ ]:
# Step 1 — min-max scale each component to [0, 1] ACROSS the full dataset
#           (both cities together so BCN and LDN scores are comparable)
scaler_risk = MinMaxScaler()
risk_features = list(WEIGHTS.keys())
scaled = scaler_risk.fit_transform(kl[risk_features].fillna(0))
scaled_df = pd.DataFrame(scaled, columns=[f'{c}_scaled' for c in risk_features], index=kl.index)

# Step 2 — weighted sum → scale to 0–100
weight_vec = np.array([WEIGHTS[c] for c in risk_features])
raw_score = scaled_df.values @ weight_vec        # (n,) float in [0, 1]
kl['risk_priority_score'] = (raw_score * 100).round(2)

print("Risk Priority Score — summary:")
print(kl['risk_priority_score'].describe().round(2))
print("\nTop 10 highest-risk subdivisions:")
print(kl.nlargest(10, 'risk_priority_score')[['city', 'geo_key', 'risk_priority_score', 'cluster_label']].to_string(index=False))

### 2c. Risk score distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
kl['risk_priority_score'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Risk Priority Score (0–100)')
axes[0].set_ylabel('Neighbourhoods')
axes[0].set_title('Risk score distribution (all cities)')

# By cluster
colours = {'saturated': '#d62728', 'emerging': '#ff7f0e', 'low_impact': '#2ca02c'}
for label, grp in kl.groupby('cluster_label'):
    grp['risk_priority_score'].hist(bins=25, ax=axes[1], alpha=0.55, label=label, color=colours[label])
axes[1].set_xlabel('Risk Priority Score (0–100)')
axes[1].set_title('Risk score by cluster label')
axes[1].legend()

plt.suptitle('Composite Risk Priority Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'risk_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved risk_score_distribution.png")

## 3. Policy simulation

### 3a. London — entire-home night cap simulation (Q7)

In [ ]:
# The neighbourhood_kpis table already contains breach_count_90/60/30 computed
# at listing level by Member 2.  We rename them to the knowledge layer schema.
#
# listings_impacted_N = count of entire-home listings in that neighbourhood
# with ttm_days_booked > N nights in the trailing 12 months.

LONDON_CAPS = [90, 60, 30]
ldn = kl[kl['city'] == 'london'].copy()

print("London policy simulation — entire-home listings impacted by night cap:")
print(f"  Cap 90 nights : {ldn['breach_count_90'].sum():,.0f} listings affected across {len(ldn)} subdivisions")
print(f"  Cap 60 nights : {ldn['breach_count_60'].sum():,.0f} listings affected")
print(f"  Cap 30 nights : {ldn['breach_count_30'].sum():,.0f} listings affected")

top_ldn = ldn.nlargest(10, 'breach_count_90')[
    ['geo_key', 'str_density', 'breach_count_90', 'breach_count_60', 'breach_count_30', 'risk_priority_score']
]
print("\nTop 10 London subdivisions by 90-night cap impact:")
print(top_ldn.to_string(index=False))

### 3b. Barcelona — RESIDE registration compliance (Q7 BCN)

In [ ]:
bcn = kl[kl['city'] == 'barcelona'].copy()

print("Barcelona — RESIDE proxy (unregistered entire-home listings):")
print(f"  Total unregistered entire-home listings: {bcn['reside_unregistered_count'].sum():,.0f}")
print(f"  Potential housing units recoverable (upper bound): {bcn['reside_unregistered_count'].sum():,.0f}")

top_bcn = bcn.nlargest(10, 'reside_unregistered_count')[
    ['geo_key', 'str_density', 'reside_unregistered_count', 'reside_unregistered_share', 'risk_priority_score']
]
print("\nTop 10 Barcelona subdivisions by unregistered count:")
print(top_bcn.to_string(index=False))

### 3c. Policy simulation bar charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# London — 90 night cap, top 15 subdivisions
ldn_top15 = ldn.nlargest(15, 'breach_count_90').sort_values('breach_count_90')
bars = axes[0].barh(ldn_top15['geo_key'], ldn_top15['breach_count_90'], color='#1f77b4')
axes[0].set_xlabel('Listings impacted (90-night cap)')
axes[0].set_title('London — Top 15 subdivisions\nentire-home listings > 90 nights/year')

# Barcelona — RESIDE unregistered, top 10
bcn_top10 = bcn[bcn['reside_unregistered_count'] > 0].nlargest(10, 'reside_unregistered_count').sort_values('reside_unregistered_count')
axes[1].barh(bcn_top10['geo_key'], bcn_top10['reside_unregistered_count'], color='#ff7f0e')
axes[1].set_xlabel('Unregistered entire-home listings (RESIDE proxy)')
axes[1].set_title('Barcelona — Top 10 subdivisions\nwithout RESIDE registration')

plt.suptitle('Policy Simulation Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'policy_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved policy_simulation.png")

## 4. Assemble the final knowledge layer

In [ ]:
# SCHEMA (from team/03_segmentation_modelling_member3.md):
# city, neighborhood, subdivision,
# str_density, entire_home_share, commercial_host_share,
# median_nightly_price, avg_occupancy, breach_rate,
# risk_priority_score, cluster_label,
# listings_impacted_90, listings_impacted_60, listings_impacted_30

# neighbourhood ≡ geo_key in this dataset (subdivision where available,
# otherwise the borough / district fallback set by Member 2's geo_key logic)

knowledge_layer = kl.copy()

# Rename to canonical schema
knowledge_layer = knowledge_layer.rename(columns={
    'geo_key':            'subdivision',    # primary geo identifier
    'breach_rate_90':     'breach_rate',    # canonical name for the chatbot
    'breach_count_90':    'listings_impacted_90',
    'breach_count_60':    'listings_impacted_60',
    'breach_count_30':    'listings_impacted_30',
})

# Add neighborhood column (= subdivision since geo_key already falls back)
knowledge_layer['neighborhood'] = knowledge_layer['subdivision']

# Select & order final columns
FINAL_COLS = [
    'city', 'neighborhood', 'subdivision', 'geo_level',
    'str_density', 'entire_home_share', 'commercial_host_share',
    'median_nightly_price', 'avg_occupancy', 'breach_rate',
    'risk_priority_score', 'cluster_label', 'cluster_distance',
    'listings_impacted_90', 'listings_impacted_60', 'listings_impacted_30',
    # Extended columns for richer chatbot answers
    'active_listings', 'entire_home_count', 'reside_unregistered_count',
    'reside_unregistered_share', 'p25_price', 'p75_price',
    'total_revenue', 'multi_listing_host_share', 'professional_management_share',
    'breach_count_60', 'breach_count_30', 'active_share',
]

knowledge_layer = knowledge_layer[[c for c in FINAL_COLS if c in knowledge_layer.columns]]

# Quality check: no entirely missing rows
print(f"Final knowledge layer shape: {knowledge_layer.shape}")
print(f"Null counts per column:")
print(knowledge_layer.isnull().sum()[knowledge_layer.isnull().sum() > 0])

## 5. Save the final knowledge layer

In [ ]:
out_path = PROCESSED_DIR / 'knowledge_layer.csv'
knowledge_layer.to_csv(out_path, index=False)
print(f"\n✅ Saved final knowledge layer → {out_path}")
print(f"   Rows: {len(knowledge_layer)}")
print(f"   Columns ({len(knowledge_layer.columns)}): {list(knowledge_layer.columns)}")
print(f"   Cities: {knowledge_layer['city'].value_counts().to_dict()}")
print(f"   Cluster labels: {knowledge_layer['cluster_label'].value_counts().to_dict()}")

## 6. Append model_metrics.md — Risk Score & Policy Simulation

In [ ]:
metrics_path = Path('..') / 'reports' / 'model_metrics.md'

ldn_90 = int(kl[kl.city=='london']['breach_count_90'].sum())
ldn_60 = int(kl[kl.city=='london']['breach_count_60'].sum())
ldn_30 = int(kl[kl.city=='london']['breach_count_30'].sum())
bcn_unreg = int(kl[kl.city=='barcelona']['reside_unregistered_count'].sum())

block = f"""

## Composite Risk Priority Score

| Component | Weight | Rationale |
|---|---|---|
| `str_density` | 0.20 | Raw STR footprint |
| `entire_home_share` | 0.30 | Housing displacement proxy (highest weight) |
| `commercial_host_share` | 0.20 | Professionalisation signal |
| `breach_rate_90` | 0.30 | Regulatory non-compliance risk (highest weight) |

Scaling: MinMaxScaler applied across the full combined dataset (both cities) so scores are directly comparable across Barcelona and London.

Score range: 0 (minimal pressure) → 100 (maximum pressure)

## Policy Simulation Results

### London — entire-home night cap (Q7)

| Cap threshold | Listings impacted |
|---|---|
| 90 nights/year | {ldn_90:,} |
| 60 nights/year | {ldn_60:,} |
| 30 nights/year | {ldn_30:,} |

### Barcelona — RESIDE compliance (Q7 BCN)

| Metric | Count |
|---|---|
| Unregistered entire-home listings | {bcn_unreg:,} |

## knowledge_layer.csv — Column documentation

| Column | Type | Definition |
|---|---|---|
| `city` | str | `barcelona` or `london` |
| `neighborhood` | str | Same as subdivision (geo_key fallback from Member 2) |
| `subdivision` | str | Primary geo key — barrio (BCN) or local neighbourhood (LDN) |
| `geo_level` | str | `subdivision` or `neighborhood` (resolution indicator) |
| `str_density` | int | Total STR listings in the subdivision |
| `entire_home_share` | float 0–1 | Fraction of listings that are entire homes |
| `commercial_host_share` | float 0–1 | Fraction of listings from commercial/super-commercial hosts |
| `median_nightly_price` | float | Median TTM average nightly rate (€/£) |
| `avg_occupancy` | float 0–1 | Mean L90D occupancy rate |
| `breach_rate` | float 0–1 | Fraction of entire-home listings > 90 booked nights/year |
| `risk_priority_score` | float 0–100 | Weighted composite risk score (Member 3) |
| `cluster_label` | str | `saturated` / `emerging` / `low_impact` (Member 3) |
| `cluster_distance` | float | Distance to cluster centroid (confidence proxy) |
| `listings_impacted_90` | int | Entire-home listings > 90 booked nights/yr |
| `listings_impacted_60` | int | Entire-home listings > 60 booked nights/yr |
| `listings_impacted_30` | int | Entire-home listings > 30 booked nights/yr |
| `active_listings` | int | Listings active in any of the past 12 months |
| `entire_home_count` | int | Count of entire-home listings |
| `reside_unregistered_count` | int | BCN: entire-home listings without RESIDE registration |
| `reside_unregistered_share` | float 0–1 | BCN: fraction of entire homes without registration |
| `p25_price`, `p75_price` | float | Nightly price quartiles |
| `total_revenue` | float | Sum of TTM revenue across all listings (€/£) |
| `multi_listing_host_share` | float 0–1 | Fraction of listings from multi-property hosts |
| `professional_management_share` | float 0–1 | Fraction managed by professional property managers |
| `breach_count_60/30` | int | Entire-home listings > 60 / 30 booked nights/yr |
| `active_share` | float 0–1 | Fraction of listings that were active in past 12 months |

"""

with open(metrics_path, 'a') as f:
    f.write(block)
print(f"Appended policy & schema docs to {metrics_path}")

## 7. Final acceptance checks

In [ ]:
print("=" * 60)
print("ACCEPTANCE CHECKS")
print("=" * 60)

# 1. Cluster labels stable
from sklearn.preprocessing import MinMaxScaler as MMS
from sklearn.cluster import KMeans as KM
from sklearn.metrics import silhouette_score as ss

CLUSTER_FEATURES = ['str_density','entire_home_share','commercial_host_share','multi_listing_host_share','avg_occupancy','breach_rate']
kl2 = pd.read_csv(PROCESSED_DIR / 'knowledge_layer.csv')
X = kl2[CLUSTER_FEATURES].fillna(0)
sc = MMS(); X_sc = sc.fit_transform(X)
km = KM(n_clusters=3, random_state=42, n_init=10)
lbl = km.fit_predict(X_sc)
sil = ss(X_sc, lbl)
print(f"1. Silhouette score: {sil:.4f} {'✓' if sil > 0.25 else '⚠ below 0.25 threshold'}")

# 2. Cluster labels stable across reruns
km2 = KM(n_clusters=3, random_state=42, n_init=10)
lbl2 = km2.fit_predict(X_sc)
stable = (lbl == lbl2).all()
print(f"2. Cluster labels stable: {'✓' if stable else '✗ UNSTABLE'}")

# 3. All rows have cluster_label
no_label = kl2['cluster_label'].isnull().sum()
print(f"3. Rows without cluster_label: {no_label} {'✓' if no_label == 0 else '✗'}")

# 4. All rows have risk_priority_score
no_score = kl2['risk_priority_score'].isnull().sum()
print(f"4. Rows without risk_priority_score: {no_score} {'✓' if no_score == 0 else '✗'}")

# 5. Policy sim reproducible
ldn_check = kl2[kl2.city=='london']['listings_impacted_90'].sum()
print(f"5. London 90-night cap total: {ldn_check:,} ✓")

# 6. knowledge_layer.csv columns
required = ['city','neighborhood','subdivision','cluster_label','risk_priority_score',
            'listings_impacted_90','listings_impacted_60','listings_impacted_30']
missing = [c for c in required if c not in kl2.columns]
print(f"6. Required columns present: {'✓' if not missing else f'MISSING: {missing}'}")

print("=" * 60)
print(f"\n✅ Notebook 07 complete. knowledge_layer.csv ready for Member 4.")
print(f"   Path: {PROCESSED_DIR / 'knowledge_layer.csv'}")
print(f"   Shape: {kl2.shape}")